# Lire, décrire et préparer un jeu de données : les maisons d'Ames

Ce TP met en pratique le chapitre « Tour d'horizon de pandas » sur des données réelles : les 1 460 ventes de maisons d'Ames (Iowa) de 2006 à 2010, décrites par 79 variables (`ames/train.csv`, chaque variable et chacun de ses codes étant expliqués dans `ames/data_description.txt`), puis, pour lire un classeur Excel, les estimations de population de l'INSEE par département (`france-population/`).

Les exercices se suivent : chacun repart du DataFrame `maisons` tel que l'ont laissé les précédents. Les Conseils supplémentaires donnent les fonctions, les paramètres et un squelette du code ; la Solution, à exécuter avant de passer à la suite si vous l'ouvrez, définit tout ce que les exercices suivants utilisent.

La cellule suivante télécharge les deux jeux de données dans le répertoire courant, celle d'après importe les bibliothèques.

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh
![ -d ames ] || bash /tmp/ghsparse.sh ames
![ -d france-population ] || bash /tmp/ghsparse.sh france-population

In [ ]:
import json
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

## 1. L'environnement de travail

Colab exécute ce notebook sur une machine virtuelle Linux qui a son propre Python et quelques centaines de paquets préinstallés (pandas, NumPy, pyarrow, matplotlib…), aux versions que Google choisit et met à jour sans prévenir ; ce que vous y installez ou écrivez disparaît à la déconnexion. Le Codespace du cours reproduit le même environnement. Pour le recréer sur votre machine, on relève les versions, puis on les impose dans un projet `uv` (frame « Environnement virtuel avec `uv` ») :

```console
$ uv init --python 3.13 analyse && cd analyse
$ uv add pandas==2.2.3 numpy==2.1.3 …    # la commande que vous allez construire
$ uv run --with jupyter jupyter lab       # ce notebook, dans cet environnement
```

Affichez la version de pandas (`pd.__version__`). Écrivez le rapport de `pd.show_versions` dans le fichier `versions.json`, relisez-le avec le module `json`, affichez la version de Python qu'il donne, et construisez le dictionnaire `versions` des versions de `pandas`, `numpy`, `pyarrow`, `openpyxl`, `xlrd` et `matplotlib`. Une valeur `None` signale un paquet absent : il n'en faut aucune, pyarrow servant à Parquet (exercice 9) et xlrd au classeur `.xls` de l'INSEE. Affichez enfin la commande `uv add` qui installe exactement ces versions.

**Documentation utile**

- [`pandas.show_versions`](https://pandas.pydata.org/docs/reference/api/pandas.show_versions.html)
- [`json.load`](https://docs.python.org/fr/3/library/json.html#json.load)

In [ ]:
# Votre code ici

### Conseils supplémentaires

`pd.show_versions(as_json="versions.json")` écrit le rapport dans ce fichier au lieu de l'afficher. Relu avec `json.load`, c'est un dictionnaire à deux clés : `"system"` (dont `"python"`, la version de Python) et `"dependencies"` (nom du paquet → version, ou `None` s'il n'est pas installé). Une compréhension de dictionnaire extrait les six paquets, et `" ".join(...)` assemble la commande à partir des chaînes `f"{nom}=={version}"` de chaque paquet (`versions.items()`), de la forme `uv add pandas==2.2.3 numpy==2.1.3 pyarrow==…` (les versions de Colab en octobre 2026).

```python
print(pd.__version__)
pd.show_versions(as_json="versions.json")
with open("versions.json") as f:
  rapport = json.load(f)
print("Python", rapport["system"]["python"])
paquets = ["pandas", "numpy", "pyarrow", "openpyxl", "xlrd", "matplotlib"]
versions = {nom: ... for nom in paquets}
print(versions)
print("uv add " + " ".join(...))
```

### Solution

In [ ]:
print(pd.__version__)
pd.show_versions(as_json="versions.json")
with open("versions.json") as f:
  rapport = json.load(f)
print("Python", rapport["system"]["python"])
paquets = ["pandas", "numpy", "pyarrow", "openpyxl", "xlrd", "matplotlib"]
versions = {nom: rapport["dependencies"][nom] for nom in paquets}
print(versions)
print("uv add " + " ".join(f"{nom}=={version}" for nom, version in versions.items()))

## 2. Lire le CSV sans perdre d'information

Dans `ames/train.csv`, une case vide est une valeur inconnue, mais `NA` veut souvent dire « aucun » : `Alley` vaut `NA` pour « No alley access », `PoolQC` pour « No Pool » (voir `ames/data_description.txt`). Or `read_csv` lit par défaut `NA` (comme `None`, `n/a`, `NaN`…) en manquant : ces absences se confondent alors avec de vrais inconnus.

- Lisez le fichier dans `maisons`, indexé par `Id`, de sorte que seules les cases vides soient des manquants, et que `MSSubClass` (un code de type de logement, pas une quantité, malgré ses valeurs 20, 60…) soit de type `category`.
- Vérifiez avec `value_counts(dropna=False)` que `Alley` a une catégorie `NA`, et comparez le nombre total de manquants de `maisons` à celui d'une lecture par défaut.
- Lisez enfin dans `apercu` les seules colonnes `Id`, `Neighborhood` et `SalePrice`, et comparez la mémoire qu'occupent les deux DataFrame (`memory_usage(deep=True)`).

**Documentation utile**

- [`pandas.read_csv`](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

`keep_default_na=False` vide la liste des chaînes lues comme manquantes, et `na_values=[""]` n'y remet que la case vide. `dtype={"MSSubClass": "category"}` fixe le type d'une colonne dès la lecture. Attendu : 357 manquants, contre 7 829 par défaut, et pour `Alley` 1 369 `NA`, 50 `Grvl` et 41 `Pave`. Avec `usecols=`, la colonne de `index_col` doit être dans la liste. `memory_usage(deep=True).sum()` : 3,7 Mo pour `maisons`, 0,1 Mo pour `apercu`.

```python
maisons = pd.read_csv(
  "ames/train.csv",
  index_col="Id",
  keep_default_na=...,
  na_values=...,
  dtype=...,
)
print(maisons["Alley"].value_counts(dropna=False))
defaut = pd.read_csv("ames/train.csv", index_col="Id")
print(maisons.isna().sum().sum(), ...)
apercu = pd.read_csv("ames/train.csv", usecols=..., index_col="Id")
print(maisons.memory_usage(deep=True).sum(), ...)
```

### Solution

In [ ]:
maisons = pd.read_csv(
  "ames/train.csv",
  index_col="Id",
  keep_default_na=False,
  na_values=[""],
  dtype={"MSSubClass": "category"},
)
print(maisons["Alley"].value_counts(dropna=False))
defaut = pd.read_csv("ames/train.csv", index_col="Id")
print(maisons.isna().sum().sum(), defaut.isna().sum().sum())
apercu = pd.read_csv(
  "ames/train.csv", usecols=["Id", "Neighborhood", "SalePrice"], index_col="Id"
)
print(maisons.memory_usage(deep=True).sum(), apercu.memory_usage(deep=True).sum())

## 3. Décrire le jeu de données

Faites le portrait de `maisons` :

- types et mémoire réelle avec `info`, puis le nombre de colonnes de chaque type (`dtypes.value_counts()`) ;
- `describe` des colonnes de texte : quel quartier (`Neighborhood`) compte le plus de ventes ?
- `describe` des colonnes `LotArea`, `GrLivArea`, `YearBuilt`, `OverallQual` et `SalePrice` : leurs ordres de grandeur, et le prix maximal ;
- le nombre de valeurs distinctes de chaque colonne (`nunique`), trié : quelles colonnes sont presque constantes (regardez le `value_counts` de `Utilities`) ? Parmi les colonnes numériques (`select_dtypes`), lesquelles n'ont qu'une douzaine de valeurs au plus, des notes ou des mois plutôt que des quantités ?

**Documentation utile**

- [`DataFrame.nunique`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.nunique.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

`info(memory_usage="deep")` compte la mémoire réelle des chaînes (3,6 Mo) : 43 colonnes `object`, 33 `int64`, 3 `float64` et 1 `category`. `describe(include="object")` donne pour chaque colonne de texte `count`, `unique`, `top` (la valeur la plus fréquente) et `freq` (son effectif) : `NAmes`, 225 ventes ; `.T` met une ligne par colonne, plus lisible. Le prix maximal est de 755 000 dollars. Trié, `nunique()` place en tête `Street`, `Utilities` et `CentralAir` (deux valeurs) : `Utilities` vaut `AllPub` pour 1 459 maisons sur 1 460, elle n'apprend rien. Parmi les nombres, `OverallQual` (une note de 1 à 10), `YrSold` ou `GarageCars` ont 10 valeurs au plus.

```python
maisons.info(memory_usage=...)
print(maisons.dtypes.value_counts())
print(maisons.describe(include=...).T)
print(
  maisons[["LotArea", "GrLivArea", "YearBuilt", "OverallQual", "SalePrice"]]
  .describe()
  .T
)
print(maisons.nunique().sort_values().head())
print(maisons["Utilities"].value_counts())
print(maisons.select_dtypes(...).nunique().sort_values().head(12))
```

### Solution

In [ ]:
maisons.info(memory_usage="deep")
print(maisons.dtypes.value_counts())
print(maisons.describe(include="object").T)
print(
  maisons[["LotArea", "GrLivArea", "YearBuilt", "OverallQual", "SalePrice"]]
  .describe()
  .T
)
print(maisons.nunique().sort_values().head())
print(maisons["Utilities"].value_counts())
print(maisons.select_dtypes("number").nunique().sort_values().head(12))

## 4. Valeurs manquantes

Il ne reste comme manquants dans `maisons` que les cases vides du fichier.

- **Profil** : la part de manquants de chaque colonne (`isna().mean()`), triée, pour les seules colonnes qui en ont.
- **Absence structurelle** : l'année du garage `GarageYrBlt` manque-t-elle exactement pour les maisons sans garage (`GarageType` vaut `"NA"`) ? Vérifiez-le avec `pd.crosstab`. Faites de même pour le sous-sol, entre `BsmtQual` et `BsmtExposure` valant `"NA"` : une maison sort de la diagonale. Laquelle (son `Id`), et que vaut alors son `NA` ? Remplacez-le par un vrai manquant, `np.nan`, en une seule affectation avec `.loc`.
- **Imputation** : la longueur de façade sur rue `LotFrontage` dépend du quartier. Remplacez ses manquants par la médiane de leur quartier (`groupby` puis `transform`), et vérifiez qu'il n'en reste aucun.

**Documentation utile**

- [`pandas.crosstab`](https://pandas.pydata.org/docs/reference/api/pandas.crosstab.html)
- [`SeriesGroupBy.transform`](https://pandas.pydata.org/docs/reference/api/pandas.api.typing.SeriesGroupBy.transform.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

`manquants[manquants > 0]` ne garde que les colonnes concernées : `LotFrontage` (17,7 %), `GarageYrBlt` (5,5 %), `MasVnrType` et `MasVnrArea` (0,5 %), `Electrical` (une maison). `pd.crosstab(a, b)` croise deux `Series` de booléens : un tableau diagonal (1 379 et 81) montre que les deux manquent ensemble, et que les 81 années manquantes n'ont rien à imputer. Pour le sous-sol, la maison hors diagonale a un sous-sol mais une exposition `NA` : un inconnu, pas une absence. Combinez les deux masques avec `~` et `&`, et lisez l'index : c'est la maison 949. `groupby("Neighborhood")["LotFrontage"].transform("median")` renvoie une `Series` alignée sur `maisons` (la médiane du quartier de chaque maison), que `fillna` accepte : la maison 8, du quartier `NWAmes`, reçoit 80. Pour entraîner un modèle, ces médianes se calculeraient sur l'entraînement seul (frame « Imputer sans fuite, combler une série »).

```python
manquants = maisons.isna().mean().sort_values(ascending=False)
print(manquants[...])
print(pd.crosstab(maisons["GarageYrBlt"].isna(), maisons["GarageType"] == "NA"))
sans_sous_sol = maisons["BsmtQual"] == "NA"
exposition_na = ...
print(pd.crosstab(sans_sous_sol, exposition_na))
print(maisons.index[...])
maisons.loc[..., "BsmtExposure"] = np.nan
mediane_quartier = maisons.groupby("Neighborhood")["LotFrontage"].transform(...)
maisons["LotFrontage"] = maisons["LotFrontage"].fillna(...)
print(maisons["LotFrontage"].isna().sum(), maisons.loc[8, "LotFrontage"])
```

### Solution

In [ ]:
manquants = maisons.isna().mean().sort_values(ascending=False)
print(manquants[manquants > 0])
print(pd.crosstab(maisons["GarageYrBlt"].isna(), maisons["GarageType"] == "NA"))
sans_sous_sol = maisons["BsmtQual"] == "NA"
exposition_na = maisons["BsmtExposure"] == "NA"
print(pd.crosstab(sans_sous_sol, exposition_na))
print(maisons.index[~sans_sous_sol & exposition_na])
maisons.loc[~sans_sous_sol & exposition_na, "BsmtExposure"] = np.nan
mediane_quartier = maisons.groupby("Neighborhood")["LotFrontage"].transform("median")
maisons["LotFrontage"] = maisons["LotFrontage"].fillna(mediane_quartier)
print(maisons["LotFrontage"].isna().sum(), maisons.loc[8, "LotFrontage"])

## 5. Types : des échelles de qualité ordonnées

`ExterQual`, `ExterCond`, `HeatingQC` et `KitchenQual` notent une qualité sur l'échelle `Po` < `Fa` < `TA` < `Gd` < `Ex` (de « Poor » à « Excellent »). En texte, `"Ex" < "Gd"` est vrai (ordre alphabétique) et un tri mélange les niveaux.

- Créez le type `qualite`, catégoriel ordonné sur cette échelle (`pd.CategoricalDtype`), et convertissez-y les quatre colonnes.
- Comptez les maisons dont la qualité extérieure est au moins bonne (`ExterQual` au moins `"Gd"`), et affichez le prix médian par niveau de `ExterQual` : il doit croître avec la qualité.
- Mesurez, sans modifier `maisons`, la mémoire qu'occupent ses colonnes `object` avant et après `astype("category")`.

**Documentation utile**

- [`pandas.CategoricalDtype`](https://pandas.pydata.org/docs/reference/api/pandas.CategoricalDtype.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

`pd.CategoricalDtype([...], ordered=True)` définit le type, du plus petit niveau au plus grand ; `maisons[colonnes].astype(qualite)` convertit plusieurs colonnes d'un coup. Une comparaison à une chaîne suit alors l'ordre des catégories : 540 maisons. Un `groupby` sur une colonne catégorielle prend `observed=True` (les seuls niveaux présents : aucune maison n'a `Po`) et range les groupes dans l'ordre de l'échelle, de 82 250 dollars (`Fa`) à 364 606,5 dollars (`Ex`). `select_dtypes("object")` donne les colonnes de texte, et `memory_usage(deep=True).sum()` leur mémoire : 3 Mo en `object`, 0,1 Mo en `category`.

```python
qualite = pd.CategoricalDtype(..., ordered=True)
colonnes = ["ExterQual", "ExterCond", "HeatingQC", "KitchenQual"]
maisons[colonnes] = ...
print((maisons["ExterQual"] >= ...).sum())
print(maisons.groupby("ExterQual", observed=True)["SalePrice"].median())
textes = maisons.select_dtypes("object")
print(textes.memory_usage(deep=True).sum(), ...)
```

### Solution

In [ ]:
qualite = pd.CategoricalDtype(["Po", "Fa", "TA", "Gd", "Ex"], ordered=True)
colonnes = ["ExterQual", "ExterCond", "HeatingQC", "KitchenQual"]
maisons[colonnes] = maisons[colonnes].astype(qualite)
print((maisons["ExterQual"] >= "Gd").sum())
print(maisons.groupby("ExterQual", observed=True)["SalePrice"].median())
textes = maisons.select_dtypes("object")
print(
  textes.memory_usage(deep=True).sum(),
  textes.astype("category").memory_usage(deep=True).sum(),
)

## 6. Chaînes : harmoniser deux colonnes

`Exterior1st` et `Exterior2nd` donnent le revêtement extérieur principal de la maison et le second (le même quand il n'y en a qu'un). La documentation leur donne les mêmes codes, mais le fichier en écrit certains autrement dans `Exterior2nd`.

- Affichez les codes de `Exterior2nd` absents de `Exterior1st`, avec leurs effectifs (`isin`), et retrouvez dans `ames/data_description.txt` le code qu'ils devraient avoir.
- Corrigez `Exterior2nd` dans `maisons` avec un dictionnaire de remplacements (`replace`), et comptez les maisons à revêtement unique (les deux colonnes égales) avant et après la correction.
- Avec l'accesseur `.str`, comptez les maisons dont l'un au moins des deux revêtements est en bois (les codes qui commencent par `Wd`).

**Documentation utile**

- [`Series.isin`](https://pandas.pydata.org/docs/reference/api/pandas.Series.isin.html)
- [`Series.replace`](https://pandas.pydata.org/docs/reference/api/pandas.Series.replace.html)
- [`Series.str.startswith`](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.startswith.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

`~maisons["Exterior2nd"].isin(maisons["Exterior1st"])` est vrai pour les codes absents de l'autre colonne : `CmentBd` (60 maisons), `Wd Shng` (38), `Brk Cmn` (7), et `Other` (1, un code que la documentation ne donne qu'à `Exterior2nd`). Ils s'écrivent `CemntBd`, `WdShing` et `BrkComm` dans `Exterior1st`. `Series.replace` avec un dictionnaire `{ancien: nouveau}` remplace des valeurs entières (`.str.replace` remplace des morceaux de chaînes). Attendu : 1 245 revêtements uniques avant, 1 323 après. `.str.startswith("Wd")` donne un masque par colonne, et `|` les combine : 263 maisons.

```python
absents = ~maisons["Exterior2nd"].isin(...)
print(maisons.loc[absents, "Exterior2nd"].value_counts())
unique_avant = (maisons["Exterior1st"] == maisons["Exterior2nd"]).sum()
corrections = {"Wd Shng": "WdShing", "CmentBd": ..., "Brk Cmn": ...}
maisons["Exterior2nd"] = ...
unique_apres = ...
print(unique_avant, unique_apres)
bois = maisons["Exterior1st"].str.startswith("Wd") | ...
print(bois.sum())
```

### Solution

In [ ]:
absents = ~maisons["Exterior2nd"].isin(maisons["Exterior1st"])
print(maisons.loc[absents, "Exterior2nd"].value_counts())
unique_avant = (maisons["Exterior1st"] == maisons["Exterior2nd"]).sum()
corrections = {"Wd Shng": "WdShing", "CmentBd": "CemntBd", "Brk Cmn": "BrkComm"}
maisons["Exterior2nd"] = maisons["Exterior2nd"].replace(corrections)
unique_apres = (maisons["Exterior1st"] == maisons["Exterior2nd"]).sum()
print(unique_avant, unique_apres)
bois = maisons["Exterior1st"].str.startswith("Wd") | maisons[
  "Exterior2nd"
].str.startswith("Wd")
print(bois.sum())

## 7. Dates : la date de vente

Le fichier donne le mois (`MoSold`) et l'année (`YrSold`) de chaque vente.

- Construisez la colonne `date_vente`, le 1er du mois de la vente, avec `pd.to_datetime`.
- Calculez dans `mensuel` le nombre de ventes et le prix médian de chaque mois (`resample("ME")`) : combien de mois couvre le jeu, et lequel compte le plus de ventes ?
- Tracez le nombre de ventes par mois et sa moyenne glissante sur 12 mois (`rolling`) : que montre chacune des deux courbes ?
- Ajoutez l'âge de la maison à la vente, `age_vente` (l'année de `date_vente` moins `YearBuilt`), et croisez les maisons de 0 ou 1 an avec les ventes de maisons neuves (`SaleType` vaut `"New"`) avec `pd.crosstab`.

**Documentation utile**

- [`pandas.to_datetime`](https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html)
- [`DataFrame.resample`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.resample.html)
- [`Resampler.aggregate`](https://pandas.pydata.org/docs/reference/api/pandas.api.typing.Resampler.aggregate.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

`pd.to_datetime` accepte un dictionnaire (ou un DataFrame) de colonnes `year`, `month` et `day`, une valeur seule valant pour toutes les lignes : `dict(year=..., month=..., day=1)`. `maisons.resample("ME", on="date_vente")` regroupe par mois sur une colonne, sans toucher à l'index ; `.agg(ventes=("SalePrice", "size"), prix_median=("SalePrice", "median"))` nomme les colonnes du résultat. Attendu : 55 mois, de janvier 2006 à juillet 2010, dont juillet 2006 en tête (67 ventes). `mensuel["ventes"].plot()` trace la série sur son index de dates, et `ax=` superpose la seconde courbe ; `rolling(12).mean()` efface la saisonnalité (un pic chaque printemps et chaque été) : environ 26 ventes par mois, sans tendance ; elle commence par 11 `NaN`, et juillet 2010, dernier mois du jeu, est incomplet (6 ventes). `.dt.year` donne l'année d'une date : 117 des 164 maisons de 0 ou 1 an sont vendues neuves.

```python
maisons["date_vente"] = pd.to_datetime(dict(year=..., month=..., day=1))
mensuel = maisons.resample("ME", on=...).agg(
  ventes=("SalePrice", "size"), prix_median=...
)
print(len(mensuel), mensuel["ventes"].idxmax(), mensuel["ventes"].max())
ax = mensuel["ventes"].plot(label="ventes par mois")
mensuel["ventes"].rolling(...).mean().plot(ax=ax, label="moyenne sur 12 mois")
ax.legend()
plt.show()
maisons["age_vente"] = ... - maisons["YearBuilt"]
print(maisons["age_vente"].describe())
print(pd.crosstab(maisons["age_vente"] <= 1, ...))
```

### Solution

In [ ]:
maisons["date_vente"] = pd.to_datetime(
  dict(year=maisons["YrSold"], month=maisons["MoSold"], day=1)
)
mensuel = maisons.resample("ME", on="date_vente").agg(
  ventes=("SalePrice", "size"), prix_median=("SalePrice", "median")
)
print(len(mensuel), mensuel["ventes"].idxmax(), mensuel["ventes"].max())
ax = mensuel["ventes"].plot(label="ventes par mois")
mensuel["ventes"].rolling(12).mean().plot(ax=ax, label="moyenne sur 12 mois")
ax.legend()
plt.show()
maisons["age_vente"] = maisons["date_vente"].dt.year - maisons["YearBuilt"]
print(maisons["age_vente"].describe())
print(pd.crosstab(maisons["age_vente"] <= 1, maisons["SaleType"] == "New"))

## 8. Analyser selon le type de variable

- **Numérique** : l'asymétrie (`skew`) du prix `SalePrice`, puis celle de son logarithme (`np.log1p`) ; tracez leurs deux histogrammes côte à côte (`plot.hist` sur une grille `plt.subplots`). Quelles colonnes numériques sont les plus asymétriques ?
- **Numérique par catégorie** : la boîte à moustaches du prix pour chaque niveau de `ExterQual` (`sns.boxplot`).
- **Catégorielle** : par quartier, le nombre de ventes, le prix médian et la surface habitable médiane (`GrLivArea`), en une seule agrégation nommée (`groupby` puis `agg`), triée par prix médian ; puis la répartition des types de logement `BldgType` de chaque quartier, en proportions (`pd.crosstab`) : quels quartiers n'ont aucune maison individuelle (`1Fam`) ?

**Documentation utile**

- [`Series.skew`](https://pandas.pydata.org/docs/reference/api/pandas.Series.skew.html)
- [`matplotlib.pyplot.subplots`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.subplots.html)
- [`DataFrameGroupBy.agg`](https://pandas.pydata.org/docs/reference/api/pandas.api.typing.DataFrameGroupBy.agg.html)
- [`pandas.crosstab`](https://pandas.pydata.org/docs/reference/api/pandas.crosstab.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

L'asymétrie du prix vaut 1,88 (une longue queue de maisons chères), celle de son logarithme 0,12 : presque symétrique. `select_dtypes("number").skew()` place en tête `MiscVal`, `PoolArea` et `LotArea`, presque toujours nulles ou petites, avec quelques valeurs énormes. `fig, axes = plt.subplots(1, 2, figsize=(10, 4))` crée deux graphiques, que `ax=axes[0]` et `ax=axes[1]` désignent. `sns.boxplot(data=maisons, x="ExterQual", y="SalePrice")` suit l'ordre des catégories de l'exercice 5, `Po` compris (vide). L'agrégation nommée s'écrit `agg(nom=("colonne", "fonction"), ...)` : `NridgHt` est le quartier le plus cher (315 000 dollars), `MeadowV` le moins cher (88 000 dollars). `normalize="index"` divise chaque ligne par son total : `Blueste`, `BrDale`, `MeadowV` et `NPkVill` n'ont que des maisons en bande (`Twnhs`, `TwnhsE`).

```python
prix = maisons["SalePrice"]
print(prix.skew(), np.log1p(prix).skew())
print(maisons.select_dtypes("number").skew().sort_values(ascending=False).head())
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
prix.plot.hist(bins=40, ax=axes[0], title="SalePrice")
np.log1p(prix).plot.hist(bins=40, ax=..., title="log1p(SalePrice)")
plt.show()
sns.boxplot(data=maisons, x=..., y=...)
plt.show()
quartiers = (
  maisons.groupby("Neighborhood")
  .agg(
    ventes=("SalePrice", "size"),
    prix_median=...,
    surface_mediane=...,
  )
  .sort_values(...)
)
print(quartiers)
repartition = pd.crosstab(maisons["Neighborhood"], maisons["BldgType"], normalize=...)
print(repartition[repartition["1Fam"] == 0])
```

### Solution

In [ ]:
prix = maisons["SalePrice"]
print(prix.skew(), np.log1p(prix).skew())
print(maisons.select_dtypes("number").skew().sort_values(ascending=False).head())
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
prix.plot.hist(bins=40, ax=axes[0], title="SalePrice")
np.log1p(prix).plot.hist(bins=40, ax=axes[1], title="log1p(SalePrice)")
plt.show()
sns.boxplot(data=maisons, x="ExterQual", y="SalePrice")
plt.show()
quartiers = (
  maisons.groupby("Neighborhood")
  .agg(
    ventes=("SalePrice", "size"),
    prix_median=("SalePrice", "median"),
    surface_mediane=("GrLivArea", "median"),
  )
  .sort_values("prix_median", ascending=False)
)
print(quartiers)
repartition = pd.crosstab(
  maisons["Neighborhood"], maisons["BldgType"], normalize="index"
)
print(repartition[repartition["1Fam"] == 0])

## 9. Autres formats : Excel et Parquet

Deux exercices indépendants : un classeur Excel tel que le publie l'INSEE, puis l'aller-retour de `maisons` par un fichier.

### 9.1. Le classeur de l'INSEE

`france-population/estim-pop-dep-sexe-gca-1975-2023.xls` donne la population de chaque département au 1er janvier, de 1975 à 2023 : une feuille de notes, `À savoir`, puis une feuille par année, avec l'en-tête sur deux lignes de la frame « Excel : feuilles et en-têtes sur plusieurs lignes » (`header=[3, 4]`, `index_col=[0, 1]`). Construisez `population`, au format long : une ligne par département et par année, avec les colonnes `code_departement` (du texte), `annee` et `population` (des entiers), la population étant le total de la feuille (colonne `Ensemble`, `Total`).

- Lisez toutes les feuilles d'années en un seul appel à `read_excel`, à partir de la liste des feuilles du classeur (`pd.ExcelFile`). Pourquoi pas `sheet_name=None` ?
- Les codes sont du texte (`01`, `2A`), sauf ceux des départements d'outre-mer, que le classeur stocke en nombres (`971`) : convertissez-les tous en `str`.
- Ne gardez que les départements : le bas de chaque feuille a des lignes de totaux (`France métropolitaine`, `DOM`…), une ligne vide et la source. Un code de département s'écrit `01` à `95`, `2A`, `2B` ou `971` à `976` (`.str.fullmatch`).
- Assemblez les années avec `pd.concat`, l'année en clé (`keys=`).

Vérifiez : 96 départements par an jusqu'en 1989, 100 de 1990 à 2013, 101 depuis 2014 (Mayotte), et 68 042 591 habitants en 2023, le total de la ligne « France métropolitaine et DOM » de la feuille. Quelle était la population de Paris (`75`) en 1975, et en 2023 ?

**Documentation utile**

- [`pandas.ExcelFile`](https://pandas.pydata.org/docs/reference/api/pandas.ExcelFile.html)
- [`Series.droplevel`](https://pandas.pydata.org/docs/reference/api/pandas.Series.droplevel.html)
- [`Series.str.fullmatch`](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.fullmatch.html)
- [`pandas.concat`](https://pandas.pydata.org/docs/reference/api/pandas.concat.html)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

`sheet_name=None` lirait aussi `À savoir`, qui n'a pas cet en-tête : `read_excel` échoue (`IndexError`). `with pd.ExcelFile(fichier) as classeur:` ouvre le classeur une fois : `classeur.sheet_names` donne ses feuilles (garder celles dont le nom est fait de chiffres : `nom.isdigit()`), et `pd.read_excel(classeur, sheet_name=annees, ...)` renvoie un `dict` nom de feuille → DataFrame. Dans chaque feuille, `feuille[("Ensemble", "Total")]` est une `Series` indexée par (code, nom) : `.droplevel(1)` ne garde que le code. `dtype=` ne s'applique pas aux colonnes d'index : convertissez l'index après la lecture, `total.index = total.index.astype(str)`. Le masque `total.index.str.fullmatch(r"\d{2,3}|2[AB]")` écarte les totaux, la ligne vide (`"nan"`) et la source. `pd.concat(totaux, keys=[...], names=["annee", "code_departement"])` empile les `Series` sous un index à deux niveaux ; il n'y a plus de `NaN`, donc `.astype("int64")`, puis `.rename("population")` et `.reset_index()` donnent le format long : 4 850 lignes. Les codes étant du texte, Paris se cherche avec `code_departement == '75'`, pas `== 75` : 2 309 074 habitants en 1975, 2 102 650 en 2023.

```python
fichier = "france-population/estim-pop-dep-sexe-gca-1975-2023.xls"
with pd.ExcelFile(fichier) as classeur:
  annees = [nom for nom in classeur.sheet_names if ...]
  feuilles = pd.read_excel(classeur, sheet_name=annees, header=[3, 4], index_col=[0, 1])
totaux = []
for feuille in feuilles.values():
  total = feuille[("Ensemble", "Total")].droplevel(1)
  total.index = ...
  totaux.append(total[total.index.str.fullmatch(...)])
population = (
  pd.concat(totaux, keys=..., names=["annee", "code_departement"])
  .astype("int64")
  .rename("population")
  .reset_index()
)
print(population.dtypes)
print(population.groupby("annee").size().loc[[1975, 1989, 1990, 2013, 2014, 2023]])
print(population.query("annee == 2023")["population"].sum())
print(population.query(...))
```

#### Solution

In [ ]:
fichier = "france-population/estim-pop-dep-sexe-gca-1975-2023.xls"
with pd.ExcelFile(fichier) as classeur:
  annees = [nom for nom in classeur.sheet_names if nom.isdigit()]
  feuilles = pd.read_excel(classeur, sheet_name=annees, header=[3, 4], index_col=[0, 1])
totaux = []
for feuille in feuilles.values():
  total = feuille[("Ensemble", "Total")].droplevel(1)
  total.index = total.index.astype(str)
  totaux.append(total[total.index.str.fullmatch(r"\d{2,3}|2[AB]")])
population = (
  pd.concat(
    totaux,
    keys=[int(annee) for annee in annees],
    names=["annee", "code_departement"],
  )
  .astype("int64")
  .rename("population")
  .reset_index()
)
print(population.dtypes)
print(population.groupby("annee").size().loc[[1975, 1989, 1990, 2013, 2014, 2023]])
print(population.query("annee == 2023")["population"].sum())
print(population.query("code_departement == '75' and annee in [1975, 2023]"))

### 9.2. L'aller-retour par un fichier

`maisons` a maintenant des colonnes catégorielles (dont des échelles ordonnées), une date et des catégories `NA`. Écrivez-le en CSV (`maisons.csv`) et en Parquet (`maisons.parquet`), et comparez la taille des deux fichiers. Relisez ensuite chacun d'eux, et comparez à `maisons` :

- les types des colonnes, dans un DataFrame à trois colonnes (`maisons`, `csv`, `parquet`) réduit aux lignes où un type diffère ;
- le nombre de manquants ;
- le contenu entier (`equals`).

Qu'est-ce que le CSV a perdu en route ?

**Documentation utile**

- [`os.path.getsize`](https://docs.python.org/fr/3/library/os.path.html#os.path.getsize)
- [`DataFrame.equals`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.equals.html)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

`to_csv` écrit l'index (`Id`) avec les colonnes : relisez avec `index_col="Id"`. `to_parquet` garde l'index, et `read_parquet` le restaure. `os.path.getsize` : 490 ko en CSV, 147 ko en Parquet. Un DataFrame construit à partir d'un dictionnaire de `Series` (les trois `dtypes`) les aligne sur leur index, les noms des colonnes. Le CSV relu a perdu six types (`MSSubClass` redevient un entier, les échelles de qualité du texte, `date_vente` du texte) et relit les catégories `NA` comme des manquants : 7 570 au lieu de 99. Le Parquet relu est identique à `maisons` (`equals` vaut `True`).

```python
maisons.to_csv("maisons.csv")
maisons.to_parquet(...)
print(os.path.getsize("maisons.csv"), ...)
csv_relu = pd.read_csv("maisons.csv", index_col=...)
parquet_relu = pd.read_parquet(...)
types = pd.DataFrame(
  {"maisons": maisons.dtypes, "csv": csv_relu.dtypes, "parquet": ...}
)
print(types[(types["csv"] != types["maisons"]) | ...])
print(maisons.isna().sum().sum(), ...)
print(csv_relu.equals(maisons), ...)
```

#### Solution

In [ ]:
maisons.to_csv("maisons.csv")
maisons.to_parquet("maisons.parquet")
print(os.path.getsize("maisons.csv"), os.path.getsize("maisons.parquet"))
csv_relu = pd.read_csv("maisons.csv", index_col="Id")
parquet_relu = pd.read_parquet("maisons.parquet")
types = pd.DataFrame(
  {"maisons": maisons.dtypes, "csv": csv_relu.dtypes, "parquet": parquet_relu.dtypes}
)
print(
  types[(types["csv"] != types["maisons"]) | (types["parquet"] != types["maisons"])]
)
print(
  maisons.isna().sum().sum(),
  csv_relu.isna().sum().sum(),
  parquet_relu.isna().sum().sum(),
)
print(csv_relu.equals(maisons), parquet_relu.equals(maisons))